In [1]:
import os
import re
import pandas as pd

In [2]:
def folder_iterator(BASE_DIR):
    files_list = []
    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            if file.lower().endswith(".tif") or file.lower().endswith(".tiff"):
                tif_path = os.path.join(root, file)
                files_list.append(tif_path)
    
    return files_list

In [27]:
path = r"C:\Users\ruben.crespo\Downloads\GlobeES"
files = os.listdir(path)
csv_file = r"C:\Users\ruben.crespo\Downloads\ecosystem_definitions_IUCN_RL_typology.csv"

df = pd.read_csv(csv_file)
print(df.head())


   biome  code   class                                          sentences  \
0    1.0   101  Boreal  Forest_consists_of_a_continuous_stand_of_trees...   
1    1.0   101  Boreal                                          Savanna).   
2    1.0   101  Boreal  Includes_primary_and_secondary_forest_habitats...   
3    1.0   101  Boreal  Distributed_across_the_high_latitudes_of_the_n...   
4    1.0   101  Boreal  Trees_are_predominantly_coniferous_(pine,_fir_...   

     category source  
0  definition   IUCN  
1  definition   IUCN  
2  definition   IUCN  
3  definition   IUCN  
4  definition   IUCN  


In [24]:
path = r"Y:\z_resources\un_gbf\global_runs\coastal_protection\2020"

In [ ]:
"""Simple way"""
files = os.listdir(path)

In [4]:
"""Pro way"""
files_list = folder_iterator(path)

In [5]:
"""filter for an specific file type"""
file_list = []
for file in os.listdir(path):
    # Iterate over all the files in the specified directory.
    if file.endswith(".tif"):
        if file not in file_list:
            # Add the file address to the list if it had not been added before.
            file_list.append(file)
print(len(file_list),file_list[:2])

69 ['GlobES-0101_20180000_1km.tif', 'GlobES-0102_20180000_1km.tif']


In [ ]:
"""Method 1: write a prefix or suffix"""
for file in file_list:
    new_filename = os.path.join(path, file + ".tif")
    print(new_filename)
    os.rename(os.path.join(path, file), new_filename)

In [ ]:
"""Method 2: prefix and inter_text"""
for index, file in enumerate(files):
    # print(os.path.join(path, file)) #locate the original files
    new_filename = os.path.join(path, ''.join([str("Arpae_precipitation_sum_of_emilia_romagna_"), file[23:27], "_", file[27:29], ".tif"]))
    print(new_filename) #locate the renaming
    print(file.basename)
    # os.rename(os.path.join(path, file), new_filename)

In [ ]:
"""Method 3: """
for filename in os.listdir(path):
    if filename.endswith(".tif"):
        new_filename = os.path.join(path, filename[:-12] + ".tif")
        print(filename)
        print(new_filename)
        os.rename(os.path.join(path, filename), new_filename)

In [ ]:
"""Method 4: replace"""
for file in file_list:
    
    new_name = file.replace("ndvi_pyrenees_mod13a3_a", "pyrenees_mod13a3_a_ndvi_1km_epsg4326_")
    new_filename = os.path.join(path, new_name)

    print(new_filename)
    os.rename(os.path.join(path, file), new_filename)



In [ ]:
"""Method 5: split and select"""
for file in file_list:
    
    # Split the old name
    parts = file.split('_')

    # Create new name
    new_name = f"pyrenees_climpy_tmin_1km_epsg4326_filled_{parts[2]}_{parts[3]}.tif"
    new_filename = os.path.join(path, new_name)

    print(new_filename)
    os.rename(os.path.join(path, file), new_filename)

In [ ]:
"""Method 6"""
title = "global_30m_forest_age_map_natural_vs_planted"

for i, file in enumerate(file_list, start=1):
    
    # keep original extension
    _, ext = os.path.splitext(file)

    # create new filename with serial number
    new_name = f"{title}_{i:03d}{ext}"   # 001, 002, 003...

    old_path = os.path.join(path, file)
    new_path = os.path.join(path, new_name)

    print(new_path)
    os.rename(old_path, new_path)

In [9]:
"""Method 7"""

pattern = re.compile(r"tile_index_1d_(\d+)")
rows = []

def fix_path(p):
    if not p.startswith("\\\\?\\"):
        return "\\\\?\\" + os.path.abspath(p)
    return p

for old_path in files_list:
    filename = os.path.basename(old_path)
    directory = os.path.dirname(old_path)

    # ✅ Skip if already in desired format
    if filename.startswith("tile_index_1d_") and filename.endswith(".tiff"):
        print(f"Already OK: {filename}")
        continue

    match = pattern.search(filename)

    # ✅ Process only expected input format
    if match:
        tile_index = match.group(1)

        new_name = f"tile_index_1d_{tile_index}_100.m_2020.tiff"
        new_path = os.path.join(directory, new_name)

        old_path = fix_path(old_path)
        new_path = fix_path(new_path)

        # print("OLD EXISTS:", os.path.exists(old_path))
        # print("DIR EXISTS:", os.path.exists(os.path.dirname(new_path)))
        os.rename(old_path, new_path)

        rows.append({
            "old_path": old_path,
            "new_path": new_path
        })
        print(f"Renamed: {filename} -> {new_name}")
    else:
        rows.append({
            "old_path": old_path
        })
        print(f"Skipped (no tile_index found): {filename}")
rename_df = pd.DataFrame(rows)
rename_df.to_csv("coastal_protection_2020_rename_registry.csv")

Renamed: gbf.aries.coastal.protection.exposurevulnerability.combined_delta_tile_index_1d_58422_100.m_2020.tiff -> tile_index_1d_58422_100.m_2020.tiff
Renamed: gbf.aries.coastal.protection.exposurevulnerability.combined_delta_tile_index_1d_52582_100.m_2020.tiff -> tile_index_1d_52582_100.m_2020.tiff
Renamed: gbf.aries.coastal.protection.exposurevulnerability.combined_delta_tile_index_1d_33943_100.m_2020.tiff -> tile_index_1d_33943_100.m_2020.tiff
Renamed: gbf.aries.coastal.protection.exposurevulnerability.combined_delta_tile_index_1d_60318_100.m_2020.tiff -> tile_index_1d_60318_100.m_2020.tiff
Renamed: gbf.aries.coastal.protection.exposurevulnerability.combined_delta_tile_index_1d_51962_100.m_2020.tiff -> tile_index_1d_51962_100.m_2020.tiff
Renamed: gbf.aries.coastal.protection.exposurevulnerability.combined_delta_tile_index_1d_56780_100.m_2020.tiff -> tile_index_1d_56780_100.m_2020.tiff
Renamed: gbf.aries.coastal.protection.exposurevulnerability.combined_delta_tile_index_1d_46379_100.m